In [1]:
import torch
# torchvision is not used in this notebook, so it can be removed
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

%config InlineBackend.figure_format = 'svg'

In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

    except RuntimeError as e:
        print(e)

In [3]:
!wget -nc --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -n ml-latest-small.zip

ratings_data = pd.read_csv("/content/ratings_small.csv")
movie_names_data = pd.read_csv('/content/movies_metadata.csv')

--2026-04-23 06:38:49--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  1.02MB/s    in 0.9s    

2026-04-23 06:38:51 (1.02 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


/tmp/ipykernel_3672/1698969703.py:5: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movie_names_data = pd.read_csv('/content/movies_metadata.csv')


In [4]:
n_movies = len(movie_names_data)
n_user = len(ratings_data['userId'].unique())

In [6]:
movie_names_data = movie_names_data.rename(columns={'id': 'movieId'})
movie_names_data['movieId'] = pd.to_numeric(movie_names_data['movieId'], errors='coerce')
ratings_data = pd.merge(ratings_data, movie_names_data, on='movieId', how='inner')

In [7]:

ratings_data.head()

,userId,movieId,rating,timestamp,adult,belongs_to_collection,budget,genres,homepage,imdb_id,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,1,1371,2.5,1260759135,False,"{'id': 1575, 'name': 'Rocky Collection', 'post...",17000000,"[{'id': 18, 'name': 'Drama'}]",NaN,tt0084602,...,1982-05-28,270000000.0,99.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,The greatest challenge.,Rocky III,False,6.6,894.0
1,1,1405,1.0,1260759203,False,NaN,546883,"[{'id': 18, 'name': 'Drama'}, {'id': 36, 'name...",NaN,tt0015881,...,1924-12-04,0.0,140.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Greed,False,7.5,25.0
2,1,2105,4.0,1260759139,False,"{'id': 2806, 'name': 'American Pie Collection'...",11000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...",NaN,tt0163651,...,1999-07-09,235483004.0,95.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,There's nothing like your first piece.,American Pie,False,6.4,2358.0
3,1,2193,2.0,1260759198,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",http://www.crownintlpictures.com/lntitles.html,tt0085980,...,1983-03-04,22587000.0,97.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,School's out...But Bobby's education has just ...,My Tutor,False,5.8,17.0
4,1,2294,2.0,1260759108,False,NaN,22000000,"[{'id': 35, 'name': 'Comedy'}]",NaN,tt0261392,...,2001-08-22,33788161.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Hollywood had it coming,Jay and Silent Bob Strike Back,False,6.4,491.0


In [8]:

from sklearn.preprocessing import LabelEncoder
import random
Y = ratings_data.rating
user_enc = LabelEncoder()
movie_enc = LabelEncoder()
X = np.array([user_enc.fit_transform(ratings_data.userId),
              movie_enc.fit_transform(ratings_data.title)]).T

In [11]:
user_enc.classes_[4], movie_enc.classes_[len(movie_enc.classes_) - 1]

(np.int64(5), 'Şaban Oğlu Şaban')

In [12]:

for x, y in zip(X[:10], Y[:10]):
    print(list(x), y)

[np.int64(0), np.int64(1691)] 2.5
[np.int64(0), np.int64(893)] 1.0
[np.int64(0), np.int64(157)] 4.0
[np.int64(0), np.int64(1434)] 2.0
[np.int64(0), np.int64(1074)] 2.0
[np.int64(0), np.int64(516)] 2.5
[np.int64(1), np.int64(2074)] 5.0
[np.int64(1), np.int64(19)] 3.0
[np.int64(1), np.int64(2549)] 4.0
[np.int64(1), np.int64(2739)] 3.0


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

In [13]:

num_users = len(X)
num_movies = len(X)

In [15]:
from keras.layers import Input, Embedding, Flatten, Dot, Dense, Activation, Dropout
from keras.models import Model

def build_model():
    movie_input = Input(shape=[1], name="Book-Input")
    movie_embedding = Embedding(n_movies+1, 15, name="Book-Embedding")(movie_input)
    movie_vec = Flatten(name="Flatten-Books")(movie_embedding)

    user_input = Input(shape=[1], name="User-Input")
    user_embedding = Embedding(n_user+1, 15, name="User-Embedding")(user_input)
    user_vec = Flatten(name="Flatten-Users")(user_embedding)

    prod = Dot(name="Dot-Product", axes=1)([user_vec, movie_vec])

    prod = Dense(32)(prod)
    prod = Activation('relu')(prod)
    prod = Dropout(0.5)(prod)

    prod = Dense(16)(prod)
    prod = Activation('relu')(prod)
    prod = Dropout(0.5)(prod)
    prod = Dense(1)(prod)


    model = Model([user_input, movie_input], prod)
    model.compile('adam', 'mean_squared_error', metrics=['accuracy'])

    return model


model = build_model()

In [16]:
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='./checkpoint.weights.h5',
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1)

history = model.fit([X_train[:, 0], X_train[:, 1]], Y_train,
            epochs=15,
            verbose=1,
            batch_size=64,
            validation_data=([X_test[:, 0], X_test[:, 1]], Y_test),
            callbacks=[model_checkpoint_callback])

Epoch 1/15
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0227 - loss: 6.9673
Epoch 1: val_loss improved from None to 1.24062, saving model to ./checkpoint.weights.h5

Epoch 1: finished saving model to ./checkpoint.weights.h5
563/563 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.0302 - loss: 4.0907 - val_accuracy: 0.0334 - val_loss: 1.2406
Epoch 2/15
556/563 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0347 - loss: 2.2640
Epoch 2: val_loss improved from 1.24062 to 1.14161, saving model to ./checkpoint.weights.h5

Epoch 2: finished saving model to ./checkpoint.weights.h5
563/563 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.0338 - loss: 2.1450 - val_accuracy: 0.0334 - val_loss: 1.1416
Epoch 3/15
561/563 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0335 - loss: 1.7512
Epoch 3: val_loss improved from 1.14161 to 1.01471, saving model to ./checkpoint.weights.h5

Epoch 3: finished saving model to ./checkpoint.weights.h5
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.033

In [17]:

X_test[:5], Y_test[:5]

(array([[  63,  520],
        [ 528, 1653],
        [  77,   80],
        [ 606, 1118],
        [ 240, 1144]]),
 4138     5.0
 34490    4.5
 5309     4.5
 41427    5.0
 15177    3.0
 Name: rating, dtype: float64)

In [18]:

predictions = model.predict([X_test[:5, 0], X_test[:5, 1]])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step


In [19]:

print(predictions,"\n\n", Y_test[:5].values)

[[4.276419 ]
 [4.60303  ]
 [4.0959225]
 [3.828019 ]
 [2.5575306]] 

 [5.  4.5 4.5 5.  3. ]


In [20]:
movie_enc.classes_[4]

'00 Schneider - Jagd auf Nihil Baxter'

In [21]:
test_user_id = 7

def extract_true_ratings(user_id_encoded, X_test):
    '''
    Extracts true ratings for a given ENCODED user ID from X_test.
    '''
    true_ratings = []
    # Iterate through X_test to find entries for the specific encoded user
    for x_encoded_user, x_encoded_movie in X_test:
        if x_encoded_user == user_id_encoded:
            # Get original user ID and movie title to query ratings_data
            original_user_id = user_enc.classes_[user_id_encoded]
            original_movie_title = movie_enc.classes_[x_encoded_movie]

            # Find the actual rating from the original ratings_data DataFrame
            rating_row = ratings_data[
                (ratings_data['userId'] == original_user_id) &
                (ratings_data['title'] == original_movie_title)
            ]
            if not rating_row.empty:
                true_ratings.append(rating_row['rating'].values[0])

    return true_ratings

# Encode the test_user_id before passing it to the function
encoded_test_user_id = user_enc.transform([test_user_id])[0]

true_ratings_for_test_user = extract_true_ratings(encoded_test_user_id, X_test)
print(f"True ratings for user {test_user_id}: {true_ratings_for_test_user}")

True ratings for user 7: [np.float64(3.0), np.float64(5.0), np.float64(4.0), np.float64(5.0), np.float64(4.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(3.0)]


In [22]:

def extract_true_ratings(user_id, X_test):

    true_ratings = list()
    for x, y in X_test:
        if x == user_id:
            rating = ratings_data[(ratings_data['userId'] == user_enc.classes_[user_id]) \
                & (ratings_data['title'] == movie_enc.classes_[y])]['rating'].values[0]
            true_ratings.append(rating)

    return true_ratings

In [23]:
def predict_ratings(user_id, X_test):
    '''
    given user id predict all ratings for movies
    '''
    user_data = ratings_data[ratings_data['userId'] == user_id]
    movie_ids, movie_names, predictions, movie_genres = list(), list(), list(), list()
    i = 0
    for _id, movie_id in X_test:
        if user_id == X_test[i][0]:
            movie_ids.append(X_test[i, 1])
            movie_names.append(movie_enc.classes_[movie_id])
            pred = model.predict([ np.array([X_test[i, 0]]), np.array([X_test[i, 1]]) ])
            predictions.append(pred[0][0])
        i += 1
    return movie_ids, movie_names, movie_genres, predictions

In [24]:

test_user_id = 7
userid_rating_data = ratings_data[ratings_data['userId'] == test_user_id]
# userid_rating_data

In [25]:
movie_ids, movie_names, movie_genres, predictions = predict_ratings(test_user_id, X_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


In [26]:
dictionary = {"user_id": [test_user_id]*len(movie_ids),
              "movie_id": movie_ids,
              "movie_name":movie_names,
              "predicted_ratings":predictions,
              "true_ratings": extract_true_ratings(test_user_id, X_test)
              }

In [27]:

prediction_dataframe = pd.DataFrame.from_dict(dictionary, orient='index').transpose()
prediction_dataframe.sort_values('predicted_ratings', ascending=False)

,user_id,movie_id,movie_name,predicted_ratings,true_ratings
4,7,1816,Sissi,4.25134,4.5
3,7,1425,My Name Is Bruce,4.196993,5.0
1,7,2484,The Tunnel,4.08526,5.0
2,7,338,Blackmail,3.882044,5.0
6,7,54,A Brief History of Time,3.763194,0.5
0,7,497,Cleopatra Jones and the Casino of Gold,3.484641,3.0
7,7,296,Beloved Enemy,3.388783,3.0
5,7,2246,The Last Castle,2.75338,3.0
